# Day 1: Setup + Data

**Goal**: Environment running, dataset downloaded, `images.csv` produced.

**Done checkpoint**:
- `data/metadata/images.csv` exists with clean rows
- `inspect_dataset()` prints stats without errors

**Runtime**: CPU is fine for Day 1.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Base project directory on your Drive
PROJECT_DIR = '/content/drive/MyDrive/Projects/VisualSearchSystem'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Change working directory to project root for all scripts
%cd {PROJECT_DIR}
print(f'Working directory: {os.getcwd()}')

## 2. Clone Repo (first time only)

In [ ]:
import os

REPO_URL = 'https://github.com/atharvagasheTAMU/VisualSearchSystem.git'  # <-- update this

# If you haven't pushed to GitHub yet, manually copy the src/ folder to Drive instead.
# Once you have GitHub set up, this cell clones it once and skips if already cloned.
if not os.path.exists(os.path.join(PROJECT_DIR, 'src')):
    !git clone {REPO_URL} /tmp/repo
    !cp -r /tmp/repo/. {PROJECT_DIR}/
    print('Repo cloned.')
else:
    # Pull latest changes
    !git pull
    print('Repo already exists, pulled latest.')

## 3. Install Dependencies

In [ ]:
# Install all project dependencies
# This takes ~3-4 minutes on first run, much faster on subsequent runs
!pip install -q \
    torch torchvision \
    transformers \
    faiss-cpu \
    Pillow \
    pandas numpy tqdm \
    pyyaml python-dotenv \
    kaggle \
    fastapi uvicorn[standard] python-multipart aiofiles \
    streamlit \
    easyocr \
    wikipedia SPARQLWrapper \
    praw \
    httpx requests scikit-learn scipy
print('Installation complete.')

## 4. Configure API Credentials

In [ ]:
# --- Option A: Use Colab Secrets (recommended) ---
# Go to: Colab sidebar > Key icon > Add secrets
# Add: KAGGLE_USERNAME, KAGGLE_KEY

from google.colab import userdata
import os
import json
from pathlib import Path

try:
    kaggle_username = userdata.get('KAGGLE_USERNAME')
    kaggle_key = userdata.get('KAGGLE_KEY')

    # Write kaggle.json
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_text(
        json.dumps({'username': kaggle_username, 'key': kaggle_key})
    )
    (kaggle_dir / 'kaggle.json').chmod(0o600)
    print('Kaggle credentials configured from Colab Secrets.')

except Exception:
    print('Colab Secrets not set. Using manual entry below.')

    # --- Option B: Manual entry ---
    # Uncomment and fill in if not using Secrets:
    # KAGGLE_USERNAME = 'your_username'
    # KAGGLE_KEY = 'your_api_key'
    # kaggle_dir = Path.home() / '.kaggle'
    # kaggle_dir.mkdir(exist_ok=True)
    # (kaggle_dir / 'kaggle.json').write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
    # (kaggle_dir / 'kaggle.json').chmod(0o600)
    pass

## 5. Download Dataset

In [ ]:
import os
from pathlib import Path

RAW_DIR = Path(PROJECT_DIR) / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = RAW_DIR / 'images'

if IMAGES_DIR.exists() and any(IMAGES_DIR.iterdir()):
    print(f'Dataset already downloaded: {len(list(IMAGES_DIR.glob("*.jpg"))):,} images found.')
else:
    print('Downloading Kaggle fashion dataset (~570MB)...')
    !kaggle datasets download -d paramaggarwal/fashion-product-images-small -p {RAW_DIR}

    print('Extracting...')
    import zipfile
    zips = list(RAW_DIR.glob('*.zip'))
    if zips:
        with zipfile.ZipFile(zips[0], 'r') as zf:
            zf.extractall(RAW_DIR)
        print('Extraction complete.')
    else:
        print('No zip file found. Check Kaggle download.')

## 6. Inspect Dataset

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path(PROJECT_DIR) / 'data' / 'raw'

# Find styles.csv
styles_csv = next(RAW_DIR.rglob('styles.csv'), None)
if styles_csv is None:
    print('styles.csv not found. Check download step.')
else:
    df = pd.read_csv(styles_csv, on_bad_lines='skip')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'\nCategory distribution:')s
    print(df['masterCategory'].value_counts())
    print(f'\nColor distribution (top 10):')
    print(df['baseColour'].value_counts().head(10))
    print(f'\nSample rows:')
    df.head(3)

## 7. Clean Dataset → images.csv

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

import yaml
with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

# Override paths to use absolute Drive paths
config['paths']['raw_images'] = str(Path(PROJECT_DIR) / 'data' / 'raw')
config['paths']['metadata_csv'] = str(Path(PROJECT_DIR) / 'data' / 'metadata' / 'images.csv')
config['paths']['metadata_dir'] = str(Path(PROJECT_DIR) / 'data' / 'metadata')

from src.preprocessing.clean import clean_dataset
df = clean_dataset(config)
df.head()

## ✅ Day 1 Done Checkpoint

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(PROJECT_DIR) / 'data' / 'metadata' / 'images.csv'
assert csv_path.exists(), 'images.csv not found!'
df = pd.read_csv(csv_path)
assert len(df) > 0, 'images.csv is empty!'

print('Day 1 COMPLETE')
print(f'  images.csv: {len(df):,} rows')
print(f'  Columns: {list(df.columns)}')
print(f'  Categories: {df["category"].nunique()} unique')